In [3]:
!pip install openaq requests pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 4.4 MB/s eta 0:00:00


In [10]:
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import os

print("Downloading and extracting real-world Air Quality dataset from UCI Repository...")

# 1. Download the zip file directly to Colab
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00360/AirQualityUCI.zip"
zip_path = "AirQualityUCI.zip"
urllib.request.urlretrieve(url, zip_path)

# 2. Extract ONLY the CSV file from the zip archive
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extract('AirQualityUCI.csv')

# 3. Read the explicitly extracted CSV file
df = pd.read_csv('AirQualityUCI.csv', sep=';', decimal=',')

# --- The rest of the cleaning pipeline remains exactly the same ---

# Clean out blank trailing rows and columns
df = df.dropna(how='all').iloc[:, :15]

# Replace the -200 missing value placeholders with NaN
df = df.replace(-200, np.nan)

# Forward-fill then backward-fill to plug gaps for the LSTM
df = df.fillna(method='ffill').fillna(method='bfill')

# Combine Date and Time into a single unified Timestamp index
df['Timestamp'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H.%M.%S')
df = df.set_index('Timestamp')
df = df.drop(columns=['Date', 'Time'])

# Rename the primary target pollutant column so Cell 3 hooks into it perfectly
df = df.rename(columns={'PT08.S1(CO)': 'Target_Pollutant'})

# Save the real-world structured dataset to your workspace
df.to_csv("ground_aqi_data.csv")

print("\nReal-world dataset successfully loaded and saved to 'ground_aqi_data.csv'!")
print(f"Total real-world hourly records loaded: {len(df)}")
print("\nPreview of the real data columns (Target, Temperature, Relative Humidity):")
print(df[['Target_Pollutant', 'T', 'RH']].head())


Real-world dataset successfully loaded and saved to 'ground_aqi_data.csv'!
Total real-world hourly records loaded: 9357

Preview of the real data columns (Target, Temperature, Relative Humidity):
                     Target_Pollutant     T    RH
Timestamp                                        
2004-03-10 18:00:00            1360.0  13.6  48.9
2004-03-10 19:00:00            1292.0  13.3  47.7
2004-03-10 20:00:00            1402.0  11.9  54.0
2004-03-10 21:00:00            1376.0  11.0  60.0
2004-03-10 22:00:00            1272.0  11.2  59.6


/tmp/ipykernel_634/3474730922.py:30: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill').fillna(method='bfill')


In [11]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [12]:
import pandas as pd
# Read just the first row to check the headers without crashing
temp_df = pd.read_csv("ground_aqi_data.csv", nrows=0)
print("Your actual column names are:")
print(temp_df.columns.tolist())

Your actual column names are:
['Timestamp', 'CO(GT)', 'Target_Pollutant', 'NMHC(GT)', 'C6H6(GT)', 'PT08.S2(NMHC)', 'NOx(GT)', 'PT08.S3(NOx)', 'NO2(GT)', 'PT08.S4(NO2)', 'PT08.S5(O3)', 'T', 'RH', 'AH']


In [15]:
df = pd.read_csv("ground_aqi_data.csv", index_col='Timestamp', parse_dates=True)

# Final safety net to ensure no NaNs crash the neural network
df = df.dropna()

# Find the index of our target variable so the model knows what to predict
target_col_name = 'Target_Pollutant'
target_index = df.columns.get_loc(target_col_name)

# Convert to numpy array and normalize (Crucial for Neural Networks)
raw_data = df.values
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(raw_data)
print(f"Data loaded and scaled. Shape: {scaled_data.shape}")

Data loaded and scaled. Shape: (9357, 13)


In [16]:
def create_sequences(data, target_idx, lookback=48, forecast=24):
    X, y = [], []
    for i in range(len(data) - lookback - forecast):
        X.append(data[i : i + lookback])
        # Predicting the target feature for the next 24 hours
        y.append(data[i + lookback : i + lookback + forecast, target_idx])
    return np.array(X), np.array(y)

print("Building sequential memory arrays...")
X, y = create_sequences(scaled_data, target_idx=target_index, lookback=48, forecast=24)

# Sequential Split (No shuffling for time-series forecasting)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, shuffle=False)
print(f"Training sequences: {X_train.shape[0]} | Validation sequences: {X_val.shape[0]}")

Building sequential memory arrays...
Training sequences: 7428 | Validation sequences: 1857


In [17]:
model = Sequential([
    # CNN Layer: Feature Extraction (detects sudden spikes in data)
    Conv1D(filters=64, kernel_size=3, activation='relu',
           input_shape=(X_train.shape[1], X_train.shape[2]), padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),

    # BiLSTM Layer 1: Forward and backward context processing
    Bidirectional(LSTM(128, return_sequences=True, kernel_regularizer=l2(0.001))),
    Dropout(0.4),

    # BiLSTM Layer 2: Deep sequential mapping
    Bidirectional(LSTM(64, return_sequences=False, kernel_regularizer=l2(0.001))),
    Dropout(0.4),

    # Fully Connected Output Layer
    Dense(64, activation='relu', kernel_regularizer=l2(0.001)),
    Dropout(0.2),
    Dense(24, activation='linear') # 24 nodes projecting the next 24 hours
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [18]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='huber', metrics=['mae'])

# Anti-overfitting callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5)

In [19]:
history = model.fit(
    X_train, y_train,
    epochs=50, # 50 epochs is solid for a hackathon timeframe
    batch_size=64,
    validation_data=(X_val, y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

Epoch 1/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 14s 22ms/step - loss: 0.2824 - mae: 0.1475 - val_loss: 0.0426 - val_mae: 0.1106 - learning_rate: 0.0010
Epoch 2/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 0.0219 - mae: 0.1155 - val_loss: 0.0119 - val_mae: 0.1101 - learning_rate: 0.0010
Epoch 3/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0115 - mae: 0.1118 - val_loss: 0.0097 - val_mae: 0.1078 - learning_rate: 0.0010
Epoch 4/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0105 - mae: 0.1092 - val_loss: 0.0090 - val_mae: 0.1032 - learning_rate: 0.0010
Epoch 5/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - loss: 0.0094 - mae: 0.1023 - val_loss: 0.0098 - val_mae: 0.1107 - learning_rate: 0.0010
Epoch 6/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0089 - mae: 0.0989 - val_loss: 0.0097 - val_mae: 0.1052 - learning_rate: 0.0010
Epoch 7/50
117/117 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - loss: 0.0085 - mae: 0.0974 - val_loss: 0.0102 - val_mae: 0.1043 - learning_rate: 0.001

In [20]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Raw predictions (still scaled 0-1)
y_pred_scaled = model.predict(X_val)

# 2. Inverse-transform: MinMaxScaler was fit on ALL 13 columns, so to invert
#    only the target column we rebuild dummy rows with the right shape.
def inverse_target(scaled_target_col, scaler, target_idx, n_features):
    dummy = np.zeros((len(scaled_target_col), n_features))
    dummy[:, target_idx] = scaled_target_col
    return scaler.inverse_transform(dummy)[:, target_idx]

n_features = X_train.shape[2]
y_val_flat   = y_val.reshape(-1)
y_pred_flat  = y_pred_scaled.reshape(-1)

y_val_real  = inverse_target(y_val_flat,  scaler, target_index, n_features)
y_pred_real = inverse_target(y_pred_flat, scaler, target_index, n_features)

mae  = mean_absolute_error(y_val_real, y_pred_real)
rmse = np.sqrt(mean_squared_error(y_val_real, y_pred_real))
r2   = r2_score(y_val_real, y_pred_real)
mape = np.mean(np.abs((y_val_real - y_pred_real) / np.clip(y_val_real, 1e-6, None))) * 100

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")
print(f"MAPE: {mape:.2f}%")

# 3. Accuracy per forecast hour (hour 1 vs hour 24 — accuracy usually drops with horizon)
y_val_2d  = y_val.reshape(-1, 24)
y_pred_2d = y_pred_scaled.reshape(-1, 24)
for h in [0, 5, 11, 17, 23]:
    real_h = inverse_target(y_val_2d[:, h],  scaler, target_index, n_features)
    pred_h = inverse_target(y_pred_2d[:, h], scaler, target_index, n_features)
    print(f"Hour +{h+1:>2}: MAE = {mean_absolute_error(real_h, pred_h):.2f}")

59/59 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step
MAE:  131.47
RMSE: 170.11
R²:   0.3098
MAPE: 12.03%
Hour + 1: MAE = 111.86
Hour + 6: MAE = 123.97
Hour +12: MAE = 131.75
Hour +18: MAE = 139.22
Hour +24: MAE = 140.58


In [21]:
import joblib

# Separate scaler just for the target column — makes deployment inverse-transform trivial
from sklearn.preprocessing import MinMaxScaler
target_scaler = MinMaxScaler()
target_scaler.fit(df[['Target_Pollutant']].values)

model.save("aqi_model.keras")
joblib.dump(scaler, "feature_scaler.pkl")
joblib.dump(target_scaler, "target_scaler.pkl")

from google.colab import files
files.download("aqi_model.keras")
files.download("feature_scaler.pkl")
files.download("target_scaler.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>